# AHR_RL: train a pre-race greening agent on the full Betfair recordings

This notebook runs the `ahr_rl` package end-to-end on **all** recordings in
`My Drive/betfair stream data/recordings`:

1. compile each `*.ndjson.gz` into a compact tape (cached back to Drive)
2. sanity baselines (do-nothing / random / rule-based scalper)
3. optional: planted-edge sanity check on synthetic races
4. PPO training (GPU if available)
5. held-out TEST evaluation vs baselines

Runtime: GPU (T4 is fine). The env simulation is CPU-bound, so more CPU cores help more than a bigger GPU.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/betfair stream data'

Mounted at /content/drive


In [2]:
!git clone -b claude/wizardly-clarke-rb37hl https://github.com/tristanhowells/AHR_RL.git /content/AHR_RL || (cd /content/AHR_RL && git pull)
%cd /content/AHR_RL
!pip -q install orjson gymnasium>=1.0
!python -m pytest -q tests/test_exchange.py

Cloning into '/content/AHR_RL'...
remote: Enumerating objects: 521, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (78/78), done.
remote: Total 521 (delta 72), reused 68 (delta 33), pack-reused 410 (from 1)
Receiving objects: 100% (521/521), 2.99 MiB | 21.73 MiB/s, done.
Resolving deltas: 100% (291/291), done.
/content/AHR_RL
..............                                                           [100%]
14 passed in 0.17s


## 1. Compile recordings -> tapes (incremental; cached on Drive)

In [3]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
TAPES = f'{DRIVE}/rl_tapes_v1'
os.makedirs(TAPES, exist_ok=True)
!python -m ahr_rl.tape "{DRIVE}/recordings" "{TAPES}" --workers {NCPU}
# local copy is much faster to read than the Drive mount
!mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
!ls /content/tapes | wc -l

[tape] skip 20260828_2015_GoldCoast_1_261565991.ndjson.gz: EOFError('Compressed file ended before the end-of-stream marker was reached')
[tape] skip 20260828_2135_GloucesterPark_1_261605686.ndjson.gz: EOFError('Compressed file ended before the end-of-stream marker was reached')
[tape] skip 20260828_2016_Wagin_1_261597385.ndjson.gz: EOFError('Compressed file ended before the end-of-stream marker was reached')
[tape] skip 20260828_2142_Newcastle_1_261610891.ndjson.gz: EOFError('Compressed file ended before the end-of-stream marker was reached')
[tape] skip 20260828_2021_Melton_1_261607475.ndjson.gz: EOFError('Compressed file ended before the end-of-stream marker was reached')
[tape] skip 20260910_1704_Bunbury_1_262184754.ndjson.gz: empty
[tape] skip 20260828_2128_Bathurst_1_261607154.ndjson.gz: EOFError('Compressed file ended before the end-of-stream marker was reached')
[tape] skip 20260910_1707_Redcliffe_1_262228246.ndjson.gz: empty
[tape] skip 20260910_1952_Penrith_1_262233102.ndjson.

## 2. Baselines on the most recent 150 races

In [4]:
import glob
from ahr_rl.evaluate import evaluate, summarise, noop_policy, make_random_policy, make_scalper_policy
tapes = sorted(glob.glob('/content/tapes/*.npz'))
recent = tapes[-150:]
for name, pol in [('do_nothing', noop_policy), ('random_1pct', make_random_policy(0.01)), ('scalper_rule', make_scalper_policy())]:
    print(name, {k: round(v, 2) for k, v in summarise(evaluate(pol, recent)).items()})

do_nothing {'races': 150, 'mean_green_$': 0.0, 'median_green_$': 0.0, 'pct_green': 100.0, 'pct_traded': 0.0, 'pct_green_when_traded': nan, 'mean_expected_$': 0.0, 'mean_realised_$': 0.0, 'total_green_$': 0.0, 'mean_turnover_$': 0.0, 'mean_trades': 0.0, 'worst_race_$': 0.0}
random_1pct {'races': 150, 'mean_green_$': -6.85, 'median_green_$': -4.91, 'pct_green': 7.33, 'pct_traded': 100.0, 'pct_green_when_traded': 7.33, 'mean_expected_$': -6.56, 'mean_realised_$': -6.54, 'total_green_$': -1026.98, 'mean_turnover_$': 212.28, 'mean_trades': 9.13, 'worst_race_$': -49.06}
scalper_rule {'races': 150, 'mean_green_$': -0.94, 'median_green_$': -0.67, 'pct_green': 16.0, 'pct_traded': 94.0, 'pct_green_when_traded': 10.64, 'mean_expected_$': -0.91, 'mean_realised_$': -0.92, 'total_green_$': -140.48, 'mean_turnover_$': 55.22, 'mean_trades': 4.49, 'worst_race_$': -5.73}


## 2b. Edge test: is there anything to trade? (run this before long RL runs)
Supervised check on the same features and date split as the agent: do they predict 10/30/60s price moves
well enough to beat the round-trip cost? One rule per family is picked on VALIDATION days and scored once
on TEST days. ~10-20 min on the full dataset; results land in `{DRIVE}/rl_runs_ahr_rl/edge`.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!pip -q install scikit-learn
!python -m ahr_rl.edge --tapes "/content/tapes/*.npz" --out "{EDGE}" --workers {NCPU}

## 2c. Feature study: do WoM / matched-price range / WAP / trade flow add signal?
Univariate signal per feature, ablation (agent features vs + engineered vs engineered only) and permutation
importance. ~20-30 min on the full dataset; results in `{DRIVE}/rl_runs_ahr_rl/features`.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.feature_study --tapes "/content/tapes/*.npz" --out "{FEAT}" --workers {NCPU}

## 2d. Signal bot: does the passive-entry signal survive realistic execution?
Trains the edge-test models on train days, then trades them inside the full simulator (latency, queue position,
funds, auto-green, commission). Horizon and threshold are chosen on validation races; the chosen config is scored
once on test races. Reuses the samples from cell 2b. Results in `{DRIVE}/rl_runs_ahr_rl/signal_bot`.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.signal_bot --tapes "/content/tapes/*.npz" --out "{BOT}" --samples "{EDGE}/samples_base.parquet" --workers {NCPU}

## 2e. Fill-mode diagnosis: is the edge-test profit an execution artefact?
Reuses the models trained in 2d and replays the same signal on the TEST races under three fill models:
`realistic` (queue behind shown size), `no_queue` (always first in queue), `touch` (any trade at our price fills
the whole order = the edge test's assumption). ~10-20 min.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.signal_bot --tapes "/content/tapes/*.npz" --out "{BOT}" --diagnose --models "{BOT}/models_base.pkl" --horizons 10,30,60 --top-pcts 0.5,1 --workers {NCPU}

## 2f. Event study: jumps and WAP displacement. Does the price keep going or come back, and can you trade it?
Two event types, one method, on every race:
* **Jumps**: the price moves >= 3 or >= 5 ticks within 5s (the race deep dives showed big moves are jumps, not trends).
* **WAP displacement**: the price sits >= 3 or >= 5 ticks away from its session volume-weighted average matched price.
  A small sample suggested drifters above their WAP come partly back ("WAP drift, fade" = back it, green later).

Only tradeable runners (spread <= 3 ticks, price <= 30). For each event it measures the move over the next 5s-5min and to
the scheduled start, and what following or fading it would have paid with latency, real book depth and commission,
against a random-time control. Rules are picked on TRAIN days and scored once on unseen days, separately per event type.
Needs only the tapes from cell 1. ~5-15 min.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
JUMPS = f'{DRIVE}/rl_runs_ahr_rl/jumps'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
!python -m ahr_rl.jump_study --tapes "/content/tapes/*.npz" --out "{JUMPS}" --workers {NCPU}
from IPython.display import Image, display
display(Image(f'{JUMPS}/jump_curves.png'))

## 2g. Market study: tradeability, signals, and how they cross
Every 10s before the off, every runner is sampled with its market share (implied probability), favouritism rank,
time to the scheduled start (and to the actual off, hindsight only), WoM, WAP gaps, price rate of change (30s, 120s)
and volume rate of change (rate, acceleration, share of money vs share of market). Sections:
* **A** tradeability by market share, rank and time (cost, depth, typical move vs cost, hit rate needed to break even)
* **B** how well each signal predicts the next 30s / 120s / to-start / to-off move
* **C** crossed features: each signal's strength inside market-share, time and busy/quiet segments, and share x time
* **D** how the signals relate to each other
* **E** can you trade it: every signal x horizon x segment cell, picked on TRAIN days, scored on unseen days
* **F** all signals and segments crossed at once in boosted-tree models of real round-trip P&L, traded on unseen days

Needs only the tapes from cell 1. Samples are cached, so a re-run only redoes the analysis. ~15-40 min.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
MKT = f'{DRIVE}/rl_runs_ahr_rl/market'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
!python -m ahr_rl.market_study --tapes "/content/tapes/*.npz" --out "{MKT}" --workers {NCPU}
from IPython.display import Image, display
for f in ('tradeability.png', 'signal_by_time.png', 'crossed.png'):
    display(Image(f'{MKT}/{f}'))

## 2h. Race study: do track, state, metro, distance, race type, class, day and time change tradeability?
Race metadata comes from the catalogues (market name "R2 1400m CL2" -> race 2, 1400m, class 2; race type; venue;
timezone -> state; local start time -> day of week, time of day). Race-level tradeability is aggregated from the
market study samples (**run 2g first**; otherwise they are rebuilt here) for runners with >= 10% market share:
cost, depth, % of time tradeable, typical move / cost, the P&L of a random trade, the P&L with perfect hindsight
of direction, and WoM signal strength. Sections: A coverage, B each metric by each category, C all categories at
once (OLS, so e.g. metro and Saturday are separated), D venue league table, E crosses. ~2-5 min.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
MKT = f'{DRIVE}/rl_runs_ahr_rl/market'
RACES = f'{DRIVE}/rl_runs_ahr_rl/races'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
!python -m ahr_rl.race_study --tapes "/content/tapes/*.npz" --catalogues "{DRIVE}/catalogues" --samples "{MKT}/samples.parquet" --out "{RACES}" --workers {NCPU}
from IPython.display import Image, display
for f in ('races_move_cost.png', 'races_tradeable.png', 'races_matched.png', 'races_venues.png'):
    if os.path.exists(f'{RACES}/{f}'):
        display(Image(f'{RACES}/{f}'))

## P1. Value against BSP: does anything beat Betfair SP on who wins?
A different kind of edge: not price moves (which pay the spread twice) but a better estimate of the winner than the
price you get. Uses the recorded winners and BSPs. **A** is BSP fair (favourite-longshot bias by price range)?
**B** is taking the exchange price at T-5m / T-2m / T-1m / start / last better than BSP (closing-line value, luck-free,
and realised)? **C** take the exchange price only when it beats Betfair's projected BSP by x% (TRAIN-picked,
HOLDOUT-scored). **D** a win model from pre-off market data + form vs BSP's own probabilities (log-loss), and BSP
"limit on close" bets placed when model probability x BSP clears a margin. Uses the form features if V2.1 attached
catalogues to the tapes. ~2-5 min.

In [5]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
BSPOUT = f'{DRIVE}/rl_runs_ahr_rl/bsp'
!python -m ahr_rl.bsp_study --tapes "/content/tapes/*.npz" --out "{BSPOUT}"

  200/1491 races (2s)
  400/1491 races (3s)
  600/1491 races (5s)
  800/1491 races (7s)
  1000/1491 races (9s)
  1200/1491 races (11s)
  1400/1491 races (13s)
1354 races with a winner and BSP, 12715 runners (391 holdout races)

A. IS BSP FAIR? back / lay everything at BSP, per $1 (after commission), by BSP range
  bsp  runners  win_rate  implied  back_roi  back_t  lay_roi  lay_t  win/implied
   <2      261    61.686   63.407    -6.627  -1.402   -4.012 -0.473        0.973
  2-3      559    39.177   40.000    -7.159  -1.498   -2.149 -0.662        0.979
  3-5     1306    25.804   25.595    -5.120  -1.297   -2.346 -1.679        1.008
  5-8     1594    15.496   15.829    -7.481  -1.498   -0.861 -0.892        0.979
 8-13     1862    10.150    9.799    -3.495  -0.563   -1.192 -1.677        1.036
13-21     1741     4.825    6.087   -25.727  -3.390    0.842  1.620        0.793
21-51     2456     3.502    3.220    -6.342  -0.638   -0.563 -1.519        1.087
  51+     2936     1.056    0.877    3

## P2. Outside prices: does a price from outside the order book lead the exchange?
Bookmaker / tote prices often move first in AU racing, but **none are recorded yet** (nothing on Drive). This cell runs
the full lead-lag + trading test on the one outside-the-book signal the recordings contain: Betfair's **projected BSP**
(it includes SP bets never shown in the book). **A** does the outside price lead the exchange (or follow it)? **B**
trading the exchange toward the outside price when the gap is large, real round trips, TRAIN-picked, HOLDOUT-scored.
To test a real bookmaker/tote feed later, set `EXTERNAL` to CSVs with columns
`market_id,selection_id,price,secs_to_start` (or `ts` epoch-ms/ISO; the catalogues give each race's start). ~5-10 min.

In [6]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
LL = f'{DRIVE}/rl_runs_ahr_rl/leadlag'
EXTERNAL = ''   # e.g. f'{DRIVE}/external/tab_prices.csv'
ext = f'--external "{EXTERNAL}" --catalogues "{DRIVE}/catalogues"' if EXTERNAL else ''
!python -m ahr_rl.leadlag_study --tapes "/content/tapes/*.npz" --out "{LL}" --workers {NCPU} {ext}

1491 races; outside price = Betfair projected BSP (no external feed given)
  200/1491 races, 61973 samples, 16s
  400/1491 races, 115886 samples, 30s
  600/1491 races, 171661 samples, 44s
  800/1491 races, 226395 samples, 58s
  1000/1491 races, 282852 samples, 73s
  1200/1491 races, 344251 samples, 88s
  1400/1491 races, 405801 samples, 103s
441633 samples from 1488 races

gap (%) distribution: {'count': 441633.0, 'mean': -1.27, 'std': 11.75, 'min': -199.29, '5%': -18.94, '25%': -6.3, '50%': -0.84, '75%': 4.55, '95%': 14.59, 'max': 191.22}

A. LEAD-LAG (within-race rank IC, t clustered by race)
gap_to_exch    : outside-vs-exchange gap -> next exchange move (+ = exchange moves toward the outside price)
outchg_to_exch : outside price's last-30s change -> next exchange move (+ = outside leads)
exchchg_to_out : exchange's last-30s change -> next outside move (+ = exchange leads)
horizon  gap_to_exch  gap_to_exch_t  outchg_to_exch  exchchg_to_out  exch_momentum
     10        0.030         

## P3. Post-sweep passive orders: earn the snap-back with a resting order instead of paying the spread
After a >= 3 / >= 5 tick sweep, rest a $10 order on the fade (snap-back) side, joining or improving the best price,
cancel if unfilled within 10/30s, exit with a 2-tick take-profit or after 60/120s. Every trade runs through the full
simulator (latency, queue position, commission); also the follow side and random-time controls. Rules picked on TRAIN
days and scored on HOLDOUT days; the best rule is re-run with "always first in the queue" as an optimistic bound.
On the 61-race sample the best rule (fade >= 5 tick drift sweeps, top-3 runners, join, 2-tick take-profit) was
slightly positive on both train and holdout, but with only 6 holdout races. The full run decides.
~20-60 min (about 1.6 ms per simulated trade; more CPUs help).

In [7]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
SWP = f'{DRIVE}/rl_runs_ahr_rl/sweep_passive'
!python -m ahr_rl.sweep_passive --tapes "/content/tapes/*.npz" --out "{SWP}" --workers {NCPU}

  100/1491 races, 82912 simulated trades, 222s
  200/1491 races, 191296 simulated trades, 502s
  300/1491 races, 273792 simulated trades, 710s
  400/1491 races, 372992 simulated trades, 953s
  500/1491 races, 453568 simulated trades, 1149s
  600/1491 races, 554016 simulated trades, 1389s
  700/1491 races, 632192 simulated trades, 1576s
  800/1491 races, 730912 simulated trades, 1816s
  900/1491 races, 823136 simulated trades, 2034s
  1000/1491 races, 896480 simulated trades, 2217s
  1100/1491 races, 992736 simulated trades, 2460s
  1200/1491 races, 1075968 simulated trades, 2669s
  1300/1491 races, 1171488 simulated trades, 2911s
  1400/1491 races, 1273088 simulated trades, 3167s

1374912 simulated trades (1487 races), fill mode realistic

FILL RATES AND P&L, all days (per $1 of matched entry, after commission)
/content/AHR_RL/ahr_rl/sweep_passive.py:254: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of

## P3b. The post-sweep fade, tuned, with a BSP exit
Only the family P3 found near break-even: fade >= 5 tick sweeps (drift and steam; >= 3 tick drifts for comparison)
with a resting order that joins the best price. Grid: entry wait 10 / 30s, take-profit 1 / 2 / 3 ticks, hold 60s /
120s / to the scheduled start, then close what is left either aggressively (as in P3) or with a **BSP bet** (P1 found
BSP is 0.3-1.2% cheaper than crossing the spread). No knowledge of the actual off time is used (P3 had a small
look-ahead there): if the race goes off first, the position goes in-play as it stands. Scored on expected P&L at BSP
odds (luck-free), worst case and realised; picked on TRAIN days, scored on HOLDOUT days. ~5-15 min.

In [8]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
P3B = f'{DRIVE}/rl_runs_ahr_rl/sweep_bsp'
!python -m ahr_rl.sweep_bsp --tapes "/content/tapes/*.npz" --out "{P3B}" --workers {NCPU}
# optimistic bound (always first in the queue):
# !python -m ahr_rl.sweep_bsp --tapes "/content/tapes/*.npz" --out "{P3B}_noqueue" --workers {NCPU} --fill-mode no_queue

  100/1491 races, 9684 simulated trades, 34s
  200/1491 races, 22302 simulated trades, 72s
  300/1491 races, 32838 simulated trades, 107s
  400/1491 races, 45108 simulated trades, 144s
  500/1491 races, 57078 simulated trades, 182s
  600/1491 races, 68346 simulated trades, 220s
  700/1491 races, 78414 simulated trades, 256s
  800/1491 races, 92178 simulated trades, 294s
  900/1491 races, 102744 simulated trades, 330s
  1000/1491 races, 111426 simulated trades, 355s
  1100/1491 races, 121236 simulated trades, 387s
  1200/1491 races, 131124 simulated trades, 417s
  1300/1491 races, 146106 simulated trades, 457s
  1400/1491 races, 157320 simulated trades, 494s
/content/AHR_RL/ahr_rl/sweep_bsp.py:206: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[c] = df[c].fillna(False)

## P3c. Forward test of the frozen P3b rule (re-run every few days)
The only rule that passed (P3b: fade >= 5 tick drift sweeps on the top 3, join the best price for 30s, 3-tick
take-profit, close the rest at BSP from the scheduled start) is now **frozen** and scored only on races recorded
from 2026-10-05, which no test has seen. One fix: "top 3" is ranked by volume matched *so far* (P3b used the
volume at the off, which a live bot can't know); section 1 re-scores the old days both ways.
Pre-registered decision on the forward days: under 300 attempts / 100 races -> CONTINUE; then mean <= 0 -> FAIL,
t >= 2 -> PASS (go to live paper trading), otherwise continue to 600 attempts and PASS only if t >= 2.
Build tapes for new recordings first (cell 1). ~2-5 min.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
!mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/   # picks up newly built tapes
P3C = f'{DRIVE}/rl_runs_ahr_rl/p3b_forward'
!python -m ahr_rl.p3b_forward --tapes "/content/tapes/*.npz" --out "{P3C}" --workers {NCPU}
from IPython.display import Image, display
if os.path.exists(f'{P3C}/forward_cumulative.png'):
    display(Image(f'{P3C}/forward_cumulative.png'))

## P4. Community strategies: the rules Betfair traders actually automate, tested
From the Bet Angel forum (Guardian bots), Caan Berry and trading guides, written as exact rules:
* **S1 WOM scalp**: final 5 min, fav / 2nd fav, WOM > 60% / 70% on one side -> offer at the best reverse price,
  1-tick profit, 4-tick stop, no entries after T-30s, green at T-15s (trade WITH or AGAINST the heavy side)
* **S2 pressure scalp**: one side has 2x the money at the best level and 1.5x at levels 2 and 3
* **S3 gap fill**: fill an empty tick in the spread on the side WOM > 70% favours
* **C control**: the same scalp at random moments on a random side
* **V volume support / resistance**, **P spoofs** (large orders behind the best price pulled without trading),
  **N late scratchings** (prices vs their reduction-factor fair value), aggressive round trips, follow vs fade
* **A arbitrage**: back or lay the whole field at the best prices
Scalps run through the full queue-aware simulator. Every rule is picked on TRAIN days and scored on HOLDOUT days.
~10-30 min.

In [9]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
COMM = f'{DRIVE}/rl_runs_ahr_rl/community'
!python -m ahr_rl.community --tapes "/content/tapes/*.npz" --out "{COMM}" --workers {NCPU}

  100/1491 races, 8171 scalps, 635 signal events, 165s
  200/1491 races, 18053 scalps, 1687 signal events, 314s
  300/1491 races, 27164 scalps, 2412 signal events, 451s
  400/1491 races, 36240 scalps, 3180 signal events, 605s
  500/1491 races, 45022 scalps, 3806 signal events, 747s
  600/1491 races, 54242 scalps, 4586 signal events, 921s
  700/1491 races, 63268 scalps, 5232 signal events, 1063s
  800/1491 races, 72768 scalps, 5994 signal events, 1214s
  900/1491 races, 82070 scalps, 6642 signal events, 1362s
  1000/1491 races, 91285 scalps, 7187 signal events, 1506s
  1100/1491 races, 100162 scalps, 7900 signal events, 1654s
  1200/1491 races, 108948 scalps, 8491 signal events, 1804s
  1300/1491 races, 118566 scalps, 9201 signal events, 1948s
  1400/1491 races, 126962 scalps, 9929 signal events, 2114s

SCALPING RULES (full simulator, fill mode 'realistic'; P&L per $1 matched, after commission)
             rule        variant runners  train_attempts_per_race  train_fill_%  train_per_fi

## P5. Lay (or back) the n-th favourite pre-off, hedge in-play
LAY side: lay the 1st..6th favourite at T-10m / T-5m / T-1m / the last pre-off moment and leave a resting BACK at
lay price x k (k = 1.1, 1.25, 1.5, 2, 3) that persists into the race. If the runner trades up to it, the position
is green on every outcome; if not (usually because it is winning), the lay is exposed.
BACK side (the mirror): back the runner and leave a resting LAY at back price / k. Winners trade down and get
hedged for a small green; losers that never trade down lose the stake.
Fill rule 'through' (a trade one tick beyond our price) is the conservative one. Uses the raw recordings, which
run into the race. Also shows each bet held to the result. Rules picked on TRAIN days, scored on HOLDOUT days.
~5-15 min (reads the raw recordings from Drive).

In [10]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
NCPU = os.cpu_count()
LAYIP = f'{DRIVE}/rl_runs_ahr_rl/lay_inplay'
!mkdir -p /content/recordings && rsync -a "{DRIVE}/recordings/" /content/recordings/
!python -m ahr_rl.lay_inplay --recordings /content/recordings --out "{LAYIP}" --workers {NCPU}

  skip 20260828_2021_Melton_1_261607475.ndjson.gz: Compressed file ended before the end-of-stream marker was reached
  skip 20260828_2015_GoldCoast_1_261565991.ndjson.gz: Compressed file ended before the end-of-stream marker was reached
  skip 20260828_2016_Wagin_1_261597385.ndjson.gz: Compressed file ended before the end-of-stream marker was reached
  skip 20260828_2128_Bathurst_1_261607154.ndjson.gz: Compressed file ended before the end-of-stream marker was reached
  skip 20260828_2135_GloucesterPark_1_261605686.ndjson.gz: Compressed file ended before the end-of-stream marker was reached
  skip 20260828_2142_Newcastle_1_261610891.ndjson.gz: Compressed file ended before the end-of-stream marker was reached
  200/1518 recordings (39s)
  400/1518 recordings (74s)
  skip 20260910_1952_Penrith_1_262233102.ndjson.gz: Compressed file ended before the end-of-stream marker was reached
  skip 20260910_2007_Kilmore_1_262233126.ndjson.gz: Compressed file ended before the end-of-stream marker was

## P6. Paired actions: every bet comes with its own unbiased hedge
Tests a simpler action spec before training on it: per runner, every 10s, the agent picks BACK pair, LAY pair or
nothing. A pair = take the price now ($10 at risk: back stake or lay liability) and at once rest the opposite bet
k ticks better (k = 1, 2, 3, 5, 8), sized to green the runner. A hedge still unfilled at the off is settled by
**close** (green at market at the scheduled start = the env's auto-green), **bsp** (the resting hedge converts to a
BSP bet at the off, Betfair's MARKET_ON_CLOSE persistence) or **hold** (runs to the result; shown for information).
A: what each action is worth by side / k / settle / time / rank. B: a hindsight oracle (the headroom an agent could
chase). C: can boosted trees on the order-book state pick the good pairs (config picked on the last quarter of TRAIN,
scored once on HOLDOUT). D: a $500 bank per holdout race: model vs random vs always-the-favourite vs oracle.
E: one k-tick pair vs a ladder of k=1 pairs (re-enter each time a hedge fills) to the same target.
~5-15 min. Writes pairs.parquet (cached) + CSVs to the output folder.

In [11]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
P6 = f'{DRIVE}/rl_runs_ahr_rl/pair_study'
!python -m ahr_rl.pair_study --tapes "/content/tapes/*.npz" --out "{P6}" --workers {NCPU}
# optimistic hedge fills (a trade AT the hedge price is enough):
# !python -m ahr_rl.pair_study --tapes "/content/tapes/*.npz" --out "{P6}" --workers {NCPU} --fill touch

  100/1491 races, 63s
  200/1491 races, 132s
  300/1491 races, 187s
  400/1491 races, 248s
  500/1491 races, 309s
  600/1491 races, 371s
  700/1491 races, 430s
  800/1491 races, 497s
  900/1491 races, 559s
  1000/1491 races, 620s
  1100/1491 races, 688s
  1200/1491 races, 748s
  1300/1491 races, 805s
  1400/1491 races, 864s

706765 decisions (runner x 10s) from 1349 races: train 960 races, holdout 389 races  [933s]

=== A. What each action is worth (all races; P&L % of the $10 at risk, race-clustered t) ===

hedge fill = through:
       hedged_pct               mean_pct                    t              
settle        bsp  close   hold      bsp  close   hold    bsp  close   hold
side k                                                                     
back 1      61.78  51.09  61.78    -9.33 -12.15 -10.86 -50.30 -58.24  -4.64
     2      52.16  40.46  52.16    -9.01 -12.45 -11.69 -44.40 -56.76  -4.53
     3      44.04  32.37  44.04    -8.71 -12.62 -11.32 -39.96 -55.96  -3.80
     5  

## 3. (optional) Planted-edge sanity check
The agent should learn to profit on synthetic races where one runner steadily shortens. If it can't, fix the learner before blaming the market.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
RUN_SYNTHETIC = False
if RUN_SYNTHETIC:
    !python -m ahr_rl.synthetic /content/synthetic 200
    !python -m ahr_rl.train --tapes "/content/synthetic/*.npz" --out /content/runs/synthetic --total-steps 500000 --n-envs {NCPU} --eval-every 25

## 4. Train on the real recordings
Chronological split: oldest ~70% of race days train, next ~15% validation (checkpoint selection), newest ~15% test (untouched until step 5).

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.train --tapes "/content/tapes/*.npz" --out "{RUN}" --total-steps 5000000 --n-envs {max(4, NCPU)} --n-steps 256 --eval-every 25 --eval-races 200

In [ ]:
import pandas as pd
log = pd.read_csv(f'{RUN}/train_log.csv')
ax = log.plot(x='steps', y=['ep_green_mean'], figsize=(10, 4), title='train: mean worst-case $ per race (last 50)')
log.dropna(subset=['val_green']).plot(x='steps', y=['val_green'], ax=ax, marker='o')

## 5. Held-out test evaluation
`mean_green_$` = average guaranteed (worst-case, after commission) profit per race. `do_nothing` scores exactly 0, so the agent has to beat 0 on unseen races to be worth running.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.report --run "{RUN}" --split test

## 5b. (optional) Challenger: continuous allocation spec, SAC vs PPO
See `docs/COMPARISON.md`. SAC benefits a lot from the GPU (many gradient steps).

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
RUN_CHALLENGERS = False
if RUN_CHALLENGERS:
    SAC_RUN = f'{DRIVE}/rl_runs_ahr_rl/sac_cont_v1'; PPOC_RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_cont_v1'
    !python -m ahr_rl.continuous --algo sac --tapes "/content/tapes/*.npz" --out "{SAC_RUN}" --total-steps 500000 --n-envs {max(4, NCPU)} --update-every 2 --eval-races 200
    !python -m ahr_rl.continuous --algo ppo --tapes "/content/tapes/*.npz" --out "{PPOC_RUN}" --total-steps 2000000 --n-envs {max(4, NCPU)} --eval-races 200
    !python -m ahr_rl.compare --name full --discrete "{RUN}" --ppo-cont "{PPOC_RUN}" --sac "{SAC_RUN}" --out "{DRIVE}/rl_runs_ahr_rl/compare"

## 6. Paper-trade live
On a machine with Betfair API credentials (not Colab):
```
export BF_USERNAME=... BF_PASSWORD=... BF_APP_KEY=... BF_CERTS=/path/to/certs
python -m ahr_rl.live.bot --model runs/ppo_v1/best.pt          # paper (default)
```

# V2 agent (built from everything the tests showed)
See `docs/V2.md`. Run cell 1 (tapes) first in a fresh session. Order: V2.1 -> V2.2 -> V2.3 -> V2.4.
The edge-test samples from cell 2b (and the long-horizon run, if you did it) are reused by the forecaster.

## V2.1 Attach catalogue form features to the tapes (once; cached on Drive)

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
EDGE_LONG = f'{DRIVE}/rl_runs_ahr_rl/edge_long'
FC = f'{DRIVE}/rl_runs_ahr_rl/forecaster_v2.pkl'
RUN_V2 = f'{DRIVE}/rl_runs_ahr_rl/ppo_v2'
NCPU = os.cpu_count()
!python -m ahr_rl.catalogue "{TAPES}" "{DRIVE}/catalogues"
!rsync -a "{TAPES}/" /content/tapes/

## V2.2 Fit the cross-fitted price-move forecaster (train days only)

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
EDGE_LONG = f'{DRIVE}/rl_runs_ahr_rl/edge_long'
FC = f'{DRIVE}/rl_runs_ahr_rl/forecaster_v2.pkl'
RUN_V2 = f'{DRIVE}/rl_runs_ahr_rl/ppo_v2'
NCPU = os.cpu_count()
samples = [f'{EDGE}/samples_base.parquet'] + ([f'{EDGE_LONG}/samples_base.parquet'] if os.path.exists(f'{EDGE_LONG}/samples_base.parquet') else [])
SAMPLES = ",".join(samples)
print(SAMPLES)
!python -m ahr_rl.forecaster --samples "{SAMPLES}" --tapes "/content/tapes/*.npz" --out "{FC}"

## V2.3 Train the v2 agent
10s decisions, trades held for minutes or to the start, v2 features + forecaster, exploration floor.
~3M steps; watch `VAL green` in the log. `do nothing` = 0 is the bar.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
EDGE_LONG = f'{DRIVE}/rl_runs_ahr_rl/edge_long'
FC = f'{DRIVE}/rl_runs_ahr_rl/forecaster_v2.pkl'
RUN_V2 = f'{DRIVE}/rl_runs_ahr_rl/ppo_v2'
NCPU = os.cpu_count()
!python -m ahr_rl.train --env v2 --forecaster "{FC}" --tapes "/content/tapes/*.npz" --out "{RUN_V2}" --total-steps 3000000 --n-envs {max(4, NCPU)} --n-steps 256 --explore-floor 0.05 --explore-anneal-frac 0.5 --type-ent-coef 0.01 --ent-coef 0.003 --eval-every 25 --eval-races 60

## V2.4 Held-out test report, segments and deployment gate
Writes `gate.json`; the live bot refuses real money unless `passed` is true.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
EDGE_LONG = f'{DRIVE}/rl_runs_ahr_rl/edge_long'
FC = f'{DRIVE}/rl_runs_ahr_rl/forecaster_v2.pkl'
RUN_V2 = f'{DRIVE}/rl_runs_ahr_rl/ppo_v2'
NCPU = os.cpu_count()
!python -m ahr_rl.report --run "{RUN_V2}" --split test